# Introduction

This notebook contains the code for merging the cleaned measurement dataset with the information on the viruses measured.

# Imports

In [28]:
# For data import
from pathlib import Path # for handling file paths
import re # for regular expressions to create valid dataframe names

# For data merging
import os # for file and directory operations

import pandas as pd # for data manipulation
import numpy as np # for numerical operations
import seaborn as sns # for data visualization
import matplotlib.pyplot as plt # for plotting
from skbio.diversity.alpha import shannon # for calculating alpha diversity
from skbio.diversity import beta_diversity # for calculating beta diversity
from skbio.stats.ordination import pcoa # for principal coordinates analysis
from scipy.stats import pearsonr # for correlation analysis

# Data Import

## Load Samplings

In [29]:
def find_folder(*path_parts):
    possible_folders = [
        Path(*path_parts),
        Path("..") / Path(*path_parts),
    ]

    for folder in possible_folders:
        if folder.exists():
            return folder

    raise FileNotFoundError(f"Folder not found: {'/'.join(path_parts)}")


def load_merged_reads_dataframes(folder_path):
    created_dataframes = {}
    
    csv_files = set(Path(folder_path).glob("*merged_reads.csv"))
    csv_files.add(Path(folder_path) / "global_merge.csv")
    
    for csv_file in sorted(csv_files):
        dataframe_name = "df_" + re.sub(r"\W|^(?=\d)", "_", csv_file.stem).lower()
        df = pd.read_csv(csv_file)
        
        # Removing leading spaces from the feature 'name'
        df["name"] = df["name"].str.lstrip()

        created_dataframes[dataframe_name] = df

    return created_dataframes


samplings_dir = find_folder("../../../generated_files")
merged_reads_dataframes = load_merged_reads_dataframes(samplings_dir)

print("Merged reads DataFrames:", merged_reads_dataframes)

Merged reads DataFrames: {'df_copenhagen_merged_reads':                name    taxid  ERR14789322  ERR14789323  ERR14789324  \
0    Mastadenovirus    10509          NaN          NaN          NaN   
1        Sequivirus    12057          NaN          NaN          NaN   
2       Sobemovirus    12137          NaN          NaN          NaN   
3       Tombusvirus    12141          NaN          NaN          NaN   
4         Tymovirus    12148          NaN          NaN          NaN   
..              ...      ...          ...          ...          ...   
225    Baldwinvirus  3153089          NaN          NaN          NaN   
226      Hodnevirus  3153200          NaN          NaN          NaN   
227      Risoevirus  3424972          NaN          NaN          NaN   
228   Margaeryvirus  3425048          NaN          NaN          NaN   
229    Nicoomyvirus  3425078          NaN         0.18          NaN   

     ERR14789325  ERR14789326  ERR14789327  ERR14789328  ERR14789329  \
0            NaN   

## Load Viruses Data

In [30]:
def load_viruses_cleaned_dataframe(folder_path):
    csv_file = Path(folder_path) / "viruses_cleaned.csv"
    return pd.read_csv(csv_file)


viruses_dir = find_folder("../generated_files/")
df_viruses = load_viruses_cleaned_dataframe(viruses_dir)

print("df_viruses:", df_viruses.shape)

df_viruses: (14465, 12)


# Merge Data

In this step, the virus data is appended to the DataFrames for each city.

In [31]:
# Get columns of df_viruses
df_viruses_columns = df_viruses.columns.tolist()
print("Columns in df_viruses:", df_viruses_columns)

Columns in df_viruses: ['virus tax id', 'host tax id', 'host name', 'realm', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'baltimore_class']


In [32]:
# Merge one DataFrame with df_viruses by name: first genus, then species, then family.
def merge_with_viruses(df, df_viruses):
    measurements = df.copy()
    measurements["name_clean"] = measurements["name"].astype(str).str.strip().str.lower()

    viruses_lookup = df_viruses.copy()
    viruses_lookup["genus_clean"] = viruses_lookup["genus"].astype(str).str.strip().str.lower()
    viruses_lookup["species_clean"] = viruses_lookup["species"].astype(str).str.strip().str.lower()
    viruses_lookup["family_clean"] = viruses_lookup["family"].astype(str).str.strip().str.lower()
    
    merged_rows = []
    selected_rows = []

    for _, row in measurements.iterrows():
        virus_name = row["name_clean"]

        virus_info = viruses_lookup[viruses_lookup["genus_clean"] == virus_name]
        if virus_info.empty:
            virus_info = viruses_lookup[viruses_lookup["species_clean"] == virus_name]
        if virus_info.empty:
            virus_info = viruses_lookup[viruses_lookup["family_clean"] == virus_name]

        if virus_info.empty:
            output_row = row.to_dict()
            for col in df_viruses_columns:
                output_row[col] = pd.NA
            merged_rows.append(output_row)
            continue

        # Keep every matching virus-host combination as a separate row.
        for _, selected_virus in virus_info.iterrows():
            output_row = row.to_dict()
            for col in df_viruses_columns:
                output_row[col] = selected_virus[col]
            merged_rows.append(output_row)

            selected_virus = selected_virus.copy()
            selected_virus["taxid"] = row["taxid"]
            selected_virus["matched_name"] = row["name"]
            selected_rows.append(selected_virus)

    merged_df = pd.DataFrame(merged_rows)
    df_selected = pd.DataFrame(selected_rows)
    df_selected = df_selected.drop(
        columns=["genus_clean", "species_clean", "family_clean"],
        errors="ignore"
    )
    df_selected = df_selected.drop_duplicates()

    merged_df = merged_df.drop(columns=["name_clean", "virus tax id"], errors="ignore")
    return merged_df, df_selected


In [33]:
# Delete existing CSV files in a given directory
def clear_csv_files(directory):
    directory = Path(directory)
    if not directory.exists():
        return

    for csv_file in directory.glob("*.csv"):
        csv_file.unlink()

In [34]:
output_folder_merged = Path("../../../generated_files/data_with_information_of_viruses/samplings_with_information_of_viruses")
output_folder_viruses_analysis = Path("../../../generated_files/data_with_information_of_viruses")
output_folder_viruses_db = Path("../../../cities/viruses")


## Create directories if they don't exist
os.makedirs(output_folder_merged, exist_ok=True)
os.makedirs(output_folder_viruses_analysis, exist_ok=True)
os.makedirs(output_folder_viruses_db, exist_ok=True)
    
## Delete existing CSV files before writing new output
clear_csv_files(output_folder_merged)
clear_csv_files(output_folder_viruses_analysis)
clear_csv_files(output_folder_viruses_db)


merged_reads_dataframes = load_merged_reads_dataframes(samplings_dir)

# Merge all merged reads DataFrames with df_viruses by merge_with_viruses function and store the results in a dictionary and in a folder
viruses_contained_df = None
merged_dataframes = {}

for name, df in merged_reads_dataframes.items():
    merged_df, df_selected = merge_with_viruses(df, df_viruses)
    
    merged_dataframes[name] = merged_df
    
    # Collect the contained viruses as DataFrame
    if not df_selected.empty:
        if viruses_contained_df is None:
            viruses_contained_df = df_selected.copy()
        else:
            viruses_contained_df = pd.concat(
                [viruses_contained_df, df_selected],
                ignore_index=True
            )

    # Save the merged DataFrame to a new CSV file
    output_file = output_folder_merged / f"{name}_with_virus_information.csv"
    merged_df.to_csv(output_file, index=False)
    
if viruses_contained_df is None:
    viruses_contained_df = pd.DataFrame()
else:
    viruses_contained_df = viruses_contained_df.drop_duplicates()
    
viruses_contained_df = viruses_contained_df.drop(columns=["virus tax id"], errors="ignore")


# Save the DataFrame with the contained viruses as a new csv file
output_file_analysis = output_folder_viruses_analysis / "viruses_contained_with_hostid.csv"
output_file_db = output_folder_viruses_db / "viruses_contained_with_hostid.csv"

viruses_contained_df.to_csv(output_file_analysis, index=False)
viruses_contained_df.to_csv(output_file_db, index=False)

# Save the DataFrame with the contained viruses without "host id" as a new csv file
viruses_contained_df = viruses_contained_df.drop(columns=["host tax id"])
viruses_contained_df = viruses_contained_df.drop_duplicates()

output_file_analysis = output_folder_viruses_analysis / "viruses_contained_without_hostid.csv"
output_file_db = output_folder_viruses_db / "viruses_contained_without_hostid.csv"

viruses_contained_df.to_csv(output_file_analysis, index=False)
viruses_contained_df.to_csv(output_file_db, index=False)